In [34]:
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from sklearn.metrics import mean_squared_error
from pathlib import Path
from openpyxl import load_workbook
import warnings
import os
warnings.filterwarnings('ignore')

# Support importing project_paths.py from the workspace root when notebooks run from subfolders.
current_path = Path.cwd().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / 'project_paths.py').exists():
        sys.path.insert(0, str(parent))
        break

from project_paths import data_path

# Create plots directory if it doesn't exist
plots_dir = 'Plots'
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)
    print(f"✅ Created '{plots_dir}' directory for saving plots")

# Define your cell's active area in cm² to convert I to J (mA/cm²)
ACTIVE_AREA_CM2 = 0.1 # this will be calculated after, is just a placeholder for now

# The code automatically inverts them to make both curves comparable.
INVERT_EXP_CURRENT = True

In [35]:
# -------------------------------------------------------------
# PLOT 1: READ EXPERIMENTAL DATA & CONVERT TO JV CURVE
# -------------------------------------------------------------

# File paths (Adjust these to match your actual file names in the folder)
exp_file = data_path('1st Round', 'IV Graph', 'CdS3_2_IV Graph.txt')
results_file = data_path('1st Round', 'Corrected', 'CdS3_2_Results Table.xlsx')
output_txt = data_path('1st Round', 'Corrected', 'CdS3_2_JV Graph.txt')

# The specific measurement you want to plot (JV Graph)
col_name = "2_CdS3_23 2026-02-09 15-44-46"

# 1. READ RESULTS TABLE TO CALCULATE EXACT ACTIVE AREA
calculated_area = ACTIVE_AREA_CM2
correction_factor = 1.0
try:
    results_df = pd.read_excel(results_file, engine='openpyxl')
    print(f"📊 Available measurements in Results Table:")
    for i, meas in enumerate(results_df['Measurement'].values):
        print(f"   {i}: {meas}")
    target_row = results_df[results_df['Measurement'] == col_name]
    if not target_row.empty:
        isc_a = target_row['Isc A'].values[0]
        jsc_ma = target_row['Jsc mA/cm2'].values[0]
        calculated_area = (isc_a / jsc_ma) * 1000
        print(f"\n✅ Target measurement '{col_name}' found in Results Table.")
        print(f"📐 Dynamically calculated Active Area: {calculated_area:.4f} cm²")
    else:
        print(f"\n⚠️ Measurement '{col_name}' NOT FOUND in Results Table!")
        print(f"   Please update 'col_name' variable to match one of the measurements above.")
        print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm²")
        calculated_area = ACTIVE_AREA_CM2
    wb = load_workbook(results_file, data_only=True)
    ws = wb.active
    rows = list(ws.iter_rows(values_only=True))
    for idx, row in enumerate(rows):
        if row and row[0] == 'Correction Factor':
            if idx + 1 < len(rows):
                cf_candidate = rows[idx + 1][0]
                if cf_candidate is not None:
                    correction_factor = float(cf_candidate)
                    print(f"\n✅ Correction factor loaded from Excel: {correction_factor}")
            break
    else:
        print("\n⚠️ Correction Factor row not found in Excel file; using 1.0")
except Exception as e:
    print(f"❌ Error reading Results Table or Correction Factor: {type(e).__name__}: {e}")
    print(f"   Using fallback area: {ACTIVE_AREA_CM2} cm² and correction factor 1.0")
    calculated_area = ACTIVE_AREA_CM2
    correction_factor = 1.0

# 2. READ IV GRAPH & CORRECT EVERY CURRENT CURVE
exp_df_raw = pd.read_csv(exp_file, sep='\t', header=None, dtype=str)
header_row2 = exp_df_raw.iloc[1].astype(str).fillna('')
current_cols = [i for i, value in enumerate(header_row2) if 'imeas' in value.lower()]
if not current_cols:
    raise ValueError(f"No current columns found in {exp_file} using the second header row")

for row_idx in range(2, exp_df_raw.shape[0]):
    for col_idx in current_cols:
        value = exp_df_raw.iat[row_idx, col_idx]
        try:
            corrected_value = float(value) * correction_factor
            exp_df_raw.iat[row_idx, col_idx] = f"{corrected_value:.8f}"
        except Exception:
            pass

# Save corrected IV file with the same original structure
output_txt = Path(output_txt)
output_txt.parent.mkdir(parents=True, exist_ok=True)
exp_df_raw.to_csv(output_txt, sep='\t', index=False, header=False)
#print(f"✅ Saved corrected IV graph file to {output_txt}")

# Locate the selected measurement for plotting
selected_col_idx = None
for i in range(exp_df_raw.shape[1]):
    if col_name in str(exp_df_raw.iat[0, i]):
        selected_col_idx = i
        break

if selected_col_idx is None:
    raise ValueError(f"Column '{col_name}' not found in {exp_file}")

# Skip the first two header rows when extracting the selected curve
v_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, selected_col_idx], errors='coerce')
i_exp_raw = pd.to_numeric(exp_df_raw.iloc[2:, selected_col_idx + 1], errors='coerce')

# Remove NaN values
valid_mask = pd.notna(v_exp_raw) & pd.notna(i_exp_raw)
v_exp = v_exp_raw[valid_mask].astype(float).values
i_exp = i_exp_raw[valid_mask].astype(float).values

# Convert Current (A) to Current Density (mA/cm2) using the dynamically calculated area
j_exp = (i_exp * 1000) / calculated_area

# Invert current if necessary (to match SCAPS typical output or 1st quadrant representation)
if INVERT_EXP_CURRENT:
    j_exp = -j_exp

exp_data = pd.DataFrame({
    'Vmeas': v_exp,
    'jtot(mA/cm2)': j_exp,
})

📊 Available measurements in Results Table:
   0: 2_CdS3_1 2026-02-09 15-06-27
   1: 2_CdS3_2 2026-02-09 15-09-05
   2: 2_CdS3_3 2026-02-09 15-10-56
   3: 2_CdS3_4 2026-02-09 15-12-27
   4: 2_CdS3_5 2026-02-09 15-13-55
   5: 2_CdS3_6 2026-02-09 15-15-24
   6: 2_CdS3_7 2026-02-09 15-17-27
   7: 2_CdS3_8 2026-02-09 15-19-11
   8: 2_CdS3_9 2026-02-09 15-20-59
   9: 2_CdS3_10 2026-02-09 15-22-29
   10: 2_CdS3_11 2026-02-09 15-24-01
   11: 2_CdS3_12 2026-02-09 15-25-26
   12: 2_CdS3_13 2026-02-09 15-27-12
   13: 2_CdS3_14 2026-02-09 15-32-45
   14: 2_CdS3_15 2026-02-09 15-34-14
   15: 2_CdS3_16 2026-02-09 15-35-55
   16: 2_CdS3_17 2026-02-09 15-37-05
   17: 2_CdS3_18 2026-02-09 15-38-38
   18: 2_CdS3_19 2026-02-09 15-40-12
   19: 2_CdS3_20 2026-02-09 15-41-10
   20: 2_CdS3_21 2026-02-09 15-42-23
   21: 2_CdS3_22 2026-02-09 15-43-24
   22: 2_CdS3_23 2026-02-09 15-44-46
   23: 2_CdS3_24 2026-02-09 15-45-54
   24: 2_CdS3_25 2026-02-09 15-47-28
   25: nan
   26: Correction Factor
   27: 0.814112

In [36]:
from openpyxl import load_workbook

results_path = Path(results_file)
workbook = load_workbook(results_path)
worksheet = workbook.active

header_row = next(
    (
        row_idx
        for row_idx in range(1, worksheet.max_row + 1)
        if worksheet.cell(row_idx, 1).value == 'Measurement'
    ),
    None,
)
if header_row is None:
    raise ValueError(f"Could not find the Measurement header in {results_path}")

headers = {
    worksheet.cell(header_row, col_idx).value: col_idx
    for col_idx in range(1, worksheet.max_column + 1)
}
required_headers = ['Isc A', 'Jsc mA/cm2', 'R at Voc', 'R at Isc']
missing_headers = [name for name in required_headers if name not in headers]
if missing_headers:
    raise ValueError(f"Missing resistance input columns in results table: {missing_headers}")

rs_col = headers.get('Rseriescm2', worksheet.max_column + 1)
rsh_col = headers.get('Rshuntcm2', max(worksheet.max_column + 1, rs_col + 1))
worksheet.cell(header_row, rs_col, 'Rseriescm2')
worksheet.cell(header_row, rsh_col, 'Rshuntcm2')

for row_idx in range(header_row + 1, worksheet.max_row + 1):
    measurement = worksheet.cell(row_idx, headers['Measurement']).value
    if measurement == 'Correction Factor':
        break
    if measurement is None:
        continue

    try:
        isc_a = float(worksheet.cell(row_idx, headers['Isc A']).value)
        jsc_ma = float(worksheet.cell(row_idx, headers['Jsc mA/cm2']).value)
        r_at_voc = float(worksheet.cell(row_idx, headers['R at Voc']).value)
        r_at_isc = float(worksheet.cell(row_idx, headers['R at Isc']).value)
        if jsc_ma == 0:
            raise ValueError('Jsc cannot be zero when calculating active area')
        active_area_cm2 = (isc_a / jsc_ma) * 1000
        rs_value = active_area_cm2 * r_at_voc
        rsh_value = active_area_cm2 * r_at_isc
    except (TypeError, ValueError, ZeroDivisionError):
        rs_value = None
        rsh_value = None

    worksheet.cell(row_idx, rs_col, rs_value)
    worksheet.cell(row_idx, rsh_col, rsh_value)

workbook.save(results_path)
workbook.close()
print(f"Saved Rs and Rsh values to {results_path}")

Saved Rs and Rsh values to C:\Users\jp_ol\OneDrive\Ambiente de Trabalho\TESE\Data_Analysis\data\1st Round\Corrected\CdS3_2_Results Table.xlsx


In [37]:
# Plot the corrected JV curve for the selected measurement (separate cell)
import os
import plotly.graph_objects as go

if 'exp_data' not in globals():
    print("exp_data not found. Run the JV-processing cell first to generate the corrected data.")
else:
    # Robust column detection (support multiple possible names)
    v_col_candidates = ['Vmeas', 'v(V)', 'V']
    j_col_candidates = ['jtot(mA/cm2)', 'jtot', 'J']
    v_col = next((c for c in v_col_candidates if c in exp_data.columns), None)
    j_col = next((c for c in j_col_candidates if c in exp_data.columns), None)
    if v_col is None or j_col is None:
        print(f"Could not find required columns in exp_data. Available columns: {list(exp_data.columns)}")
    else:
        x = exp_data[v_col].astype(float)
        y = exp_data[j_col].astype(float)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x,
            y=y,
            mode='lines',
            name=str(col_name),
            line=dict(color='royalblue', width=2)
        ))
        fig.update_layout(
            title=f"Corrected JV Curve ({col_name})",
            xaxis_title='Voltage (V)',
            yaxis_title='Current Density (mA/cm²)',
            template='plotly_white',
            width=900,
            height=600,
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
        )
        fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, showgrid=True, gridcolor='lightgray')
        fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True, showgrid=True, gridcolor='lightgray')
        # Show plot
        try:
            fig.show()
        except Exception as e:
            print('Could not render interactive figure in this environment:', e)

        """
        # Attempt to save as PNG; fallback to HTML if image export unavailable
        out_dir = plots_dir if 'plots_dir' in globals() else 'Plots'
        os.makedirs(out_dir, exist_ok=True)
        png_path = os.path.join(out_dir, f"{str(col_name).replace(' ', '_')}_Corrected_JV.png")
        html_path = os.path.join(out_dir, f"{str(col_name).replace(' ', '_')}_Corrected_JV.html")
        try:
            fig.write_image(png_path, width=900, height=600)
            print(f"Saved corrected JV plot to {png_path}")
        except Exception as e:
            print('PNG export failed, saving HTML instead:', e)
            try:
                fig.write_html(html_path)
                print(f"Saved interactive plot to {html_path}")
            except Exception as e2:
                print('Failed to save plot:', e2)"""